# Module 01: NumPy for Machine Learning
## Notebook 01: Array Basics, Creation Routines, and Memory Architecture

NumPy (**Numerical Python**) is the foundational numerical computing engine in Python. High-level ML frameworks such as **Scikit-Learn, PyTorch, and TensorFlow** rely on NumPy's contiguous buffer architecture and C-level execution speed.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Explain the architectural differences between Python lists and contiguous NumPy `ndarray`s.
2. Initialize 1D, 2D, and multi-dimensional arrays using creation routines (`zeros`, `ones`, `linspace`, `eye`).
3. Inspect and manage array memory attributes (`shape`, `ndim`, `dtype`, `itemsize`, `nbytes`).
4. Select appropriate precision (`float32` vs `float64`) for machine learning workloads.
5. Work with **Structured Arrays** for composite feature records in contiguous memory.
6. Handle out-of-core datasets with **Memory-Mapped Arrays (`np.memmap`)**.
7. Utilize **Strided Memory Views (`sliding_window_view`)** for zero-copy feature extraction.

### 1. Why NumPy for Machine Learning?

In Machine Learning, datasets are represented as matrices and tensors:
- **Tabular Data**: 2D array of shape $(N, D)$ where $N$ is samples and $D$ is features.
- **Images**: 3D or 4D arrays of shape $(H, W, C)$ or $(B, H, W, C)$ (Batch, Height, Width, Channels).
- **Time Series / NLP**: 3D arrays of shape $(B, T, D)$ (Batch, Timesteps, Embedding Dimensions).

Standard Python lists store pointers to Python objects scattered across the heap. NumPy arrays store elements in a **contiguous block of memory** with a uniform data type.
- **Cache locality**: The CPU cache pre-fetches contiguous memory, making access orders of magnitude faster.
- **SIMD vectorization**: Modern CPUs apply Single Instruction Multiple Data operations on contiguous chunks simultaneously.
- **Zero interpreter overhead**: Computation executes in compiled C/Fortran routines without per-element type dispatch.

In [ ]:
import numpy as np
import sys
import time

print(f"NumPy version: {np.__version__}")

#### Performance Comparison: Python List vs. NumPy Array

Squaring 1,000,000 numbers using standard Python list comprehensions vs. NumPy vectorized operations:

In [ ]:
# Squaring 1,000,000 numbers using standard Python list
size = 1_000_000
python_list = list(range(size))

start_time = time.time()
squared_list = [x ** 2 for x in python_list]
list_duration = time.time() - start_time

# Squaring using NumPy array
numpy_array = np.arange(size)

start_time = time.time()
squared_array = numpy_array ** 2
numpy_duration = time.time() - start_time

print(f"Python list comprehension time: {list_duration:.4f} seconds")
print(f"NumPy vectorized operation time: {numpy_duration:.4f} seconds")
print(f"NumPy speedup factor: {list_duration / numpy_duration:.1f}x faster!")

---
### 2. Creating Arrays from Existing Data

The foundational way to create an `ndarray` is using `np.array()` with Python lists or nested lists.

In [ ]:
# 1D Array (Vector) - e.g., target labels y in regression
vector_1d = np.array([1.5, 2.7, 3.9, 4.1, 5.0])
print("1D Array:")
print(vector_1d)
print(f"Shape: {vector_1d.shape}, Dimensions: {vector_1d.ndim}")

# 2D Array (Matrix) - e.g., feature matrix X with 3 samples and 4 features
matrix_2d = np.array([
    [10, 20, 30, 40],
    [15, 25, 35, 45],
    [12, 22, 32, 42]
])
print("\n2D Feature Matrix X:")
print(matrix_2d)
print(f"Shape: {matrix_2d.shape}, Dimensions: {matrix_2d.ndim}")

#### Saving and Loading Numerical CSV Files in NumPy

While Pandas is preferred for mixed-type tables, NumPy provides high-speed disk I/O routines for pure numerical matrices:
- **`np.savetxt()`**: Exports 1D or 2D numerical arrays directly to CSV or text files with custom delimiters.
- **`np.loadtxt()`**: High-speed loader for clean, uniformly-formatted numeric text/CSV files.
- **`np.genfromtxt()`**: Robust loader capable of handling missing values and column names.

In [ ]:
import os

# Robust path detection for repository data_files directory
data_dir = "data_files" if os.path.exists("data_files") else "../data_files"
sensor_csv = os.path.join(data_dir, "sensor_telemetry.csv")

# 1. Load pre-existing numeric sensor telemetry CSV into a NumPy array
sensor_data = np.loadtxt(sensor_csv, delimiter=",", skiprows=1)
print(f"Loaded sensor telemetry directly from {sensor_csv}:")
print(f"Array shape: {sensor_data.shape} (Samples x Features)")
print("First 3 sensor samples (timestamp, temp, humidity, pressure, vibration):\n", sensor_data[:3])

# 2. Save a transformed slice back to CSV using np.savetxt()
demo_export_csv = os.path.join(data_dir, "exported_sensor_slice.csv")
np.savetxt(demo_export_csv, sensor_data[:5], delimiter=",", fmt="%.3f", header="t,temp,hum,press,vib", comments="")
print(f"\nExported 5 samples to CSV: {demo_export_csv}")

# Clean up demonstration export file
if os.path.exists(demo_export_csv):
    os.remove(demo_export_csv)

---
### 3. Built-in Array Creation Routines

NumPy provides specialized functions to generate arrays of specific shapes and patterns without manually constructing Python lists.

#### A. Constant Value Arrays (`zeros`, `ones`, `full`, `empty`)
- `np.zeros(shape)`: Zero-initializing weight tensors or gradient accumulators.
- `np.ones(shape)`: Bias feature columns ($x_0 = 1$) for linear regression.
- `np.full(shape, value)`: Initialize arrays with any arbitrary scalar baseline.
- `np.empty(shape)`: Allocates uninitialized memory buffer without clearing it (fastest allocation, but contains garbage values).

In [ ]:
# Zeros array of shape (3, 3)
zero_weights = np.zeros(shape=(3, 3), dtype=np.float32)
print("Zeros (3x3):\n", zero_weights)

# Ones array of shape (2, 4)
bias_vector = np.ones(shape=(2, 4), dtype=np.float32)
print("\nOnes (2x4):\n", bias_vector)

# Full array with a specific constant (e.g., initial baseline = 42.0)
filled_array = np.full(shape=(2, 3), fill_value=42.0)
print("\nFull with 42.0 (2x3):\n", filled_array)

# Empty array (uninitialized memory allocation)
uninitialized = np.empty(shape=(2, 2))
print("\nEmpty (uninitialized memory):\n", uninitialized)

#### B. Numerical Sequences: `arange` vs `linspace`

- `np.arange(start, stop, step)`: Half-open interval `[start, stop)` with a given step.
- `np.linspace(start, stop, num)`: Closed interval `[start, stop]` with an exact count of evenly spaced samples.

> **Best Practice for Machine Learning:**
> When generating continuous parameter values (e.g. learning rate grids or plotting decision boundaries), always prefer `np.linspace` to avoid floating-point step rounding errors.

In [ ]:
# np.arange with integer step
int_seq = np.arange(0, 20, 2)
print("arange [0, 20, step=2]:", int_seq)

# np.linspace for parameter grids
# E.g., learning rate candidate values between 0.001 and 0.1
learning_rates = np.linspace(0.001, 0.1, num=10)
print("\nlinspace 10 values in [0.001, 0.1]:\n", learning_rates)

# Decision boundary evaluation grid
grid_points = np.linspace(-5.0, 5.0, num=5)
print("\nGrid points [-5, 5]:", grid_points)

#### C. Identity and Diagonal Matrices

- `np.eye(N)`: Identity matrix $I_N$, essential for $L_2$ regularization (Ridge penalty: $(X^T X + \lambda I)^{-1}$).
- `np.diag(v)`: Extracts a diagonal from a matrix or constructs a diagonal matrix from a 1D vector.

In [ ]:
# Identity matrix of size 4x4
identity_matrix = np.eye(4)
print("Identity Matrix (4x4):\n", identity_matrix)

# Constructing a diagonal weight matrix
diagonal_weights = np.diag([1.5, 3.0, 0.5])
print("\nDiagonal Matrix from vector [1.5, 3.0, 0.5]:\n", diagonal_weights)

---
### 4. Essential Array Metadata & Memory Attributes

Understanding array memory layout helps diagnose shape mismatches and memory bottlenecks in large datasets.

Key attributes:
| Attribute | Description | ML Context |
|---|---|---|
| `arr.ndim` | Number of dimensions | 1 = vector, 2 = matrix, 3+ = tensor |
| `arr.shape` | Tuple of dimensions | $(N_{samples}, N_{features})$ |
| `arr.size` | Total number of elements | Product of all dimensions |
| `arr.dtype` | Data type of elements | Numerical precision |
| `arr.itemsize` | Bytes per single element | E.g. float64 = 8 bytes, float32 = 4 bytes |
| `arr.nbytes` | Total memory consumed | `arr.size * arr.itemsize` |

In [ ]:
sample_tensor = np.ones((100, 28, 28), dtype=np.float32)

print(f"Number of dimensions (.ndim):    {sample_tensor.ndim}")
print(f"Array shape (.shape):            {sample_tensor.shape}")
print(f"Total elements (.size):          {sample_tensor.size:,}")
print(f"Data type (.dtype):              {sample_tensor.dtype}")
print(f"Bytes per element (.itemsize):   {sample_tensor.itemsize} bytes")
print(f"Total array memory (.nbytes):    {sample_tensor.nbytes:,} bytes ({sample_tensor.nbytes / 1024:.2f} KB)")

---
### 5. Data Types and Type Casting (`.astype()`)

In machine learning:
- **`float64` (double precision)**: Default in NumPy. High precision, but doubles memory usage and memory bandwidth overhead.
- **`float32` (single precision)**: Industry standard in Deep Learning and large-scale tabular modeling. Halves memory footprint with negligible loss in convergence accuracy.
- **`int32` / `int64`**: Used for discrete class labels, sample indices, and cluster IDs.
- **`bool_`**: Used for boolean masks and condition filtering.

In [ ]:
# Downcasting float64 to float32
default_arr = np.array([1.0, 2.5, 3.8])
print("Default dtype: ", default_arr.dtype, f"({default_arr.itemsize} bytes/elem)")

float32_arr = default_arr.astype(np.float32)
print("Downcasted:    ", float32_arr.dtype, f"({float32_arr.itemsize} bytes/elem)")

# Converting continuous probabilities into binary prediction labels (0 or 1)
probabilities = np.array([0.15, 0.72, 0.49, 0.91, 0.33])
binary_predictions = (probabilities >= 0.5).astype(np.int32)
print("\nProbabilities:       ", probabilities)
print("Binary Predictions:  ", binary_predictions)

---
### 6. Advanced Usages: Structured Arrays, Memory-Mapping, and Strides

Beyond simple numerical arrays, production machine learning systems leverage NumPy's underlying C-memory architecture for specialized, high-performance workflows.

#### A. Structured Arrays (Composite Records in Contiguous C Memory)
In standard Python, a dataset with strings, numbers, and vectors requires dictionaries or object lists, incurring massive pointer dereferencing overhead.
NumPy's **Structured Arrays** allow you to define heterogeneous C-style structs where records are packed contiguously in memory with named fields and sub-array shapes!

In [ ]:
# Define a custom structured dtype for ML samples:
# - sample_id: 8-character string ('U8')
# - embedding: 4-dimensional float32 feature vector ('4f4')
# - label: 32-bit integer ('i4')
# - confidence: 32-bit float ('f4')
sample_dtype = np.dtype([
    ('sample_id', 'U8'),
    ('embedding', 'f4', (4,)),
    ('label', 'i4'),
    ('confidence', 'f4')
])

# Allocate contiguous array of 3 structured samples
dataset = np.zeros(3, dtype=sample_dtype)

dataset[0] = ('SMP-001', [0.12, -0.45, 0.88, 0.05], 1, 0.94)
dataset[1] = ('SMP-002', [0.75, 0.10, -0.32, 0.41], 0, 0.88)
dataset[2] = ('SMP-003', [-0.20, 0.65, 0.15, -0.90], 1, 0.98)

print("Structured Dataset Array:\n", dataset)
print("\nDirect field access (all embeddings):\n", dataset['embedding'])
print("Shape of embeddings sub-tensor:", dataset['embedding'].shape)
print("Memory per structured record:   ", dataset.itemsize, "bytes")

#### B. Memory-Mapped Arrays (`np.memmap` for Out-of-Core Datasets)

What happens when your training dataset is 50 GB, but your machine only has 16 GB of RAM?
`np.memmap` creates a NumPy array mapped directly to a binary file on disk. The OS kernel reads and writes pages on-demand without loading the entire file into physical memory.

In [ ]:
import os

# Simulate writing a large out-of-core binary matrix to disk
filename = 'large_mmap_dataset.dat'
shape = (1000, 50)  # 1000 samples, 50 features
dtype = np.float32

# 1. Create and write to memory-mapped file
mmap_write = np.memmap(filename, dtype=dtype, mode='w+', shape=shape)
mmap_write[:] = np.arange(shape[0] * shape[1]).reshape(shape)
mmap_write.flush()  # Ensure data is flushed to disk

# 2. Read only a specific mini-batch from disk on demand (mode='r')
mmap_read = np.memmap(filename, dtype=dtype, mode='r', shape=shape)
batch_slice = mmap_read[100:105, :5]  # Reads ONLY this small chunk into RAM!

print("Memory-mapped batch slice [100:105, :5]:\n", batch_slice)
print(f"Is slice a memory-mapped view? {isinstance(mmap_read, np.memmap)}")

# Clean up temporary disk file
del mmap_write, mmap_read
if os.path.exists(filename):
    os.remove(filename)

#### C. Zero-Copy Sliding Windows (`sliding_window_view`)

In signal processing, time series feature extraction, and convolutional operations, you often extract sliding windows of length $W$ with step $S$.
Creating these windows with standard slicing or loops creates millions of duplicate copies.
NumPy's `sliding_window_view` manipulates **array strides** to create an $N$-dimensional sliding window view **without copying a single byte of memory**!

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

# Continuous time series sensor signal (12 timesteps)
time_series = np.array([10.0, 12.0, 15.0, 14.0, 18.0, 20.0, 22.0, 21.0, 25.0, 28.0, 30.0, 32.0])

# Generate rolling windows of size 4
# Stride tricks creates shape: (num_windows, window_size) = (9, 4)
windows = sliding_window_view(time_series, window_shape=4)

print("Original Time Series:\n", time_series)
print("\nZero-Copy Sliding Windows (window=4):\n", windows)
print(f"\nOriginal array memory: {time_series.nbytes} bytes")
print(f"Windows virtual memory: {windows.nbytes} bytes (shares memory with original: {windows.base is time_series})")

# Efficient vectorized moving statistics across all windows simultaneously!
rolling_means = np.mean(windows, axis=1)
rolling_stds = np.std(windows, axis=1)
print("\nVectorized Moving Averages: ", np.round(rolling_means, 2))
print("Vectorized Moving Std Devs: ", np.round(rolling_stds, 2))

### Summary & Next Steps
In this notebook, you progressed from array basics to advanced memory architectures:
- Architectural foundations: contiguous buffers, SIMD vectorization, cache locality.
- Creation routines: `zeros`, `ones`, `arange`, `linspace`, `eye`.
- Metadata inspection and precision management (`float32` vs `float64`).
- **Structured Arrays**: packing heterogeneous fields into contiguous C records.
- **Memory Mapping (`np.memmap`)**: training on out-of-core datasets larger than RAM.
- **Strided Sliding Windows**: zero-copy rolling window feature extraction.

**Next Notebook:** `02_indexing_slicing_and_reshaping.ipynb` — Multi-dimensional tensor slicing, views vs copies, channel permuting, and Einstein summation (`np.einsum`).